<a href="https://colab.research.google.com/github/2410072/Python-Lesson/blob/main/RF_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# センサ由来データを用いた機械学習パイプライン
# Dataset : Breast Cancer Wisconsin Diagnostic (sklearn 同梱)
# Model   : Random Forest + SHAP による解釈性分析
# 環境    : Google Colab (ローカル保存なし)
# ============================================================

# Colab環境にSHAPライブラリをインストール（-qオプションで出力を抑制）
!pip install -q shap

# 数値計算用ライブラリNumPyをインポート
import numpy as np
# 表形式データ操作用ライブラリPandasをインポート
import pandas as pd
# 基本的な作図ライブラリMatplotlibをインポート
import matplotlib.pyplot as plt
# 統計的可視化ライブラリSeabornをインポート
import seaborn as sns

# scikit-learn同梱の乳がん診断データセット読み込み関数をインポート
from sklearn.datasets import load_breast_cancer
# 訓練データとテストデータへの分割関数をインポート
from sklearn.model_selection import train_test_split
# ランダムフォレスト分類器をインポート
from sklearn.ensemble import RandomForestClassifier
# 決定木構造の可視化関数をインポート
from sklearn.tree import plot_tree
# 分類性能評価用の各種指標をインポート
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# モデル解釈ライブラリSHAPをインポート
import shap

# 再現性確保のための乱数シード値を定数として定義
RANDOM_STATE = 42
# NumPyの乱数シードを固定
np.random.seed(RANDOM_STATE)

# ============================================================
# 1. データセットの読み込み（メモリ上に直接ロード）
# ============================================================
# 乳がん診断データセットをDataFrame形式で読み込み（as_frame=Trueにより返り値をPandas形式に指定）
data = load_breast_cancer(as_frame=True)
# 説明変数（特徴量）を取り出す（形状: 569サンプル × 30特徴量）
X = data.data
# 目的変数（クラスラベル）を取り出す（0: 悪性, 1: 良性）
y = data.target
# クラス名（'malignant', 'benign'）を取り出す
target_names = data.target_names

# データセットのサンプル数と特徴量数を表示
print(f"サンプル数: {X.shape[0]}, 特徴量数: {X.shape[1]}")
# クラスIDとクラス名の対応を表示
print(f"クラス: {dict(enumerate(target_names))}")

# ============================================================
# 2. データセット読み込み直後の可視化
# ============================================================
# 説明変数のDataFrameを複製して可視化用DataFrameを作成
df_full = X.copy()
# 目的変数（数値ラベル）を列として追加
df_full["target"] = y
# 数値ラベルを文字列ラベル（'malignant'/'benign'）に変換した列を追加
df_full["diagnosis"] = df_full["target"].map({0: target_names[0], 1: target_names[1]})

# --- 2.1 先頭サンプルおよびデータ型の確認 ---
# セクション見出しを表示
print("\n=== データフレーム先頭5件 ===")
# 先頭5行を表示してデータの概観を確認
print(df_full.head())
# セクション見出しを表示
print("\n=== データ型と非欠損数 ===")
# 各列のデータ型と非欠損値数を表示
df_full.info()

# --- 2.2 欠損値の可視化（ヒートマップ） ---
# 図のサイズを指定して新規Figureを作成
plt.figure(figsize=(12, 4))
# 欠損値の有無をヒートマップで可視化
sns.heatmap(
    df_full.isnull(),       # 欠損か否かのブール値DataFrame（True=欠損）
    cbar=False,             # カラーバーは非表示
    yticklabels=False,      # 行ラベルは煩雑になるため非表示
    cmap="viridis"          # 配色テーマ（黄=欠損, 紫=非欠損）
)
# グラフタイトルを設定
plt.title("Missing Value Heatmap (yellow = missing)")
# レイアウトを自動調整
plt.tight_layout()
# プロットを描画
plt.show()
# 欠損値の総数を集計して表示
print(f"欠損値の総数: {df_full.isnull().sum().sum()}")

# --- 2.3 クラス分布の可視化 ---
# 図のサイズを指定して新規Figureを作成
plt.figure(figsize=(5, 4))
# クラスごとのサンプル数を棒グラフで描画
sns.countplot(
    x="diagnosis",                     # 横軸に診断ラベル
    data=df_full,                      # 描画対象のDataFrame
    palette=["salmon", "steelblue"]    # クラス毎の配色
)
# グラフタイトルを設定
plt.title("Class Distribution")
# レイアウトを自動調整
plt.tight_layout()
# プロットを描画
plt.show()

# --- 2.4 全30特徴量のクラス別箱ひげ図 ---
# target列を削除し、long形式（縦持ち）に変換して描画用DataFrameを生成
df_melt = df_full.drop(columns=["target"]).melt(
    id_vars="diagnosis",   # 識別変数として診断ラベルを保持
    var_name="feature",    # 特徴量名を格納する列名
    value_name="value"     # 値を格納する列名
)
# 各特徴量のスケールを揃えるため、特徴量ごとにz-score標準化を適用
df_melt["value"] = df_melt.groupby("feature")["value"].transform(
    lambda v: (v - v.mean()) / v.std()   # 平均0・標準偏差1に変換するラムダ関数
)

# 図のサイズを指定して新規Figureを作成
plt.figure(figsize=(16, 7))
# 全特徴量を横軸に並べたクラス別箱ひげ図を描画
sns.boxplot(
    x="feature",                       # 横軸に特徴量名
    y="value",                         # 縦軸に標準化後の値
    hue="diagnosis",                   # クラスごとに色分け
    data=df_melt,                      # 描画対象のlong形式DataFrame
    palette=["salmon", "steelblue"],   # クラス毎の配色
    fliersize=2                        # 外れ値マーカーのサイズ
)
# 横軸ラベルを90度回転させて可読性を確保
plt.xticks(rotation=90)
# グラフタイトルを設定
plt.title("Standardized Feature Distributions by Class (All 30 Features)")
# レイアウトを自動調整
plt.tight_layout()
# プロットを描画
plt.show()

# ============================================================
# 3. 特徴量の絞り込み（解釈性確保のため "mean" 系特徴量に限定）
# ============================================================
# 列名が "mean " で始まる特徴量のみを抽出してリスト化
mean_cols = [c for c in X.columns if c.startswith("mean ")]
# 抽出した特徴量のみを含むDataFrameを生成
X_sel = X[mean_cols].copy()
# 選択された特徴量数とその名称を表示
print(f"\n選択された特徴量 ({len(mean_cols)}個): {mean_cols}")

# --- 3.1 ペアプロットによる多変量分布の俯瞰 ---
# 描画コスト低減のため代表的な5特徴量を選定
pairplot_cols = ["mean radius", "mean texture", "mean perimeter",
                 "mean smoothness", "mean concavity"]
# ペアプロット用DataFrameを抽出
df_pair = X_sel[pairplot_cols].copy()
# クラスラベルを付与
df_pair["diagnosis"] = df_full["diagnosis"]

# 散布図行列（ペアプロット）を描画
sns.pairplot(
    df_pair,                                       # 描画対象DataFrame
    hue="diagnosis",                               # クラスごとに色分け
    palette=["salmon", "steelblue"],               # クラス毎の配色
    diag_kind="kde",                               # 対角成分にカーネル密度推定を表示
    plot_kws={"alpha": 0.6, "s": 20}               # 散布図の透過度と点サイズ
)
# 全体タイトルを設定（y=1.02でプロット上部に配置）
plt.suptitle("Pair Plot of Representative Features", y=1.02)
# プロットを描画
plt.show()

# ============================================================
# 4. 統計的分析（記述統計と相関分析）
# ============================================================
# --- 4.1 記述統計量 ---
# セクション見出しを表示
print("\n=== 記述統計量 ===")
# 平均・標準偏差・分位点等の記述統計量を転置して表示
print(X_sel.describe().T)

# --- 4.2 相関分析（ヒートマップ） ---
# 特徴量間のピアソン相関係数行列を計算
corr_matrix = X_sel.corr()
# 図のサイズを指定して新規Figureを作成
plt.figure(figsize=(10, 8))
# 相関係数行列をヒートマップで可視化
sns.heatmap(
    corr_matrix,                  # 描画対象の相関行列
    annot=True,                   # 各セルに数値を表示
    fmt=".2f",                    # 数値表示は小数点以下2桁
    cmap="coolwarm",              # 配色（寒色: 負相関, 暖色: 正相関）
    vmin=-1,                      # カラーバー下限
    vmax=1,                       # カラーバー上限
    square=True,                  # セルを正方形に固定
    cbar_kws={"shrink": 0.8}      # カラーバーのサイズ調整
)
# グラフタイトルを設定
plt.title("Correlation Matrix of Selected Features")
# レイアウトを自動調整
plt.tight_layout()
# プロットを描画
plt.show()

# 上三角行列のみを抽出するためのマスクを作成し、long形式に変換して相関ペア一覧を生成
strong_corr = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))  # 上三角のみ残す
    .stack()                                                                   # MultiIndex Seriesに変換
    .reset_index()                                                             # DataFrame化
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "corr"})  # 列名を整形
)
# |r| > 0.8 を満たす強相関ペアのみ抽出し、相関の絶対値降順でソート
strong_corr = strong_corr[strong_corr["corr"].abs() > 0.8] \
    .sort_values("corr", key=abs, ascending=False)
# セクション見出しを表示
print("\n=== 強相関ペア (|r| > 0.8) ===")
# 抽出された強相関ペアを表示
print(strong_corr.to_string(index=False))

# ============================================================
# 5. ランダムフォレストによる分類
# ============================================================
# データを訓練用とテスト用に分割
X_train, X_test, y_train, y_test = train_test_split(
    X_sel,                       # 説明変数
    y,                           # 目的変数
    test_size=0.25,              # テストデータの割合（25%）
    stratify=y,                  # クラス比率を保持した層化抽出
    random_state=RANDOM_STATE    # 乱数シード（再現性確保）
)

# ランダムフォレスト分類器を初期化
rf_model = RandomForestClassifier(
    n_estimators=200,            # 構築する決定木の本数
    max_depth=5,                 # 各木の最大深度（過学習抑制）
    random_state=RANDOM_STATE,   # 乱数シード（再現性確保）
    n_jobs=-1                    # 並列計算に全CPUコアを使用
)
# 訓練データでモデルを学習
rf_model.fit(X_train, y_train)

# テストデータに対する予測ラベルを取得
y_pred = rf_model.predict(X_test)
# テスト精度（Accuracy）を計算して表示
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}")
# セクション見出しを表示
print("\n=== Classification Report ===")
# 適合率・再現率・F1スコア等の詳細評価指標を表示
print(classification_report(
    y_test,                      # 真値ラベル
    y_pred,                      # 予測ラベル
    target_names=target_names    # クラス名（表示用）
))

# 混同行列を計算
cm = confusion_matrix(y_test, y_pred)
# 図のサイズを指定して新規Figureを作成
plt.figure(figsize=(5, 4))
# 混同行列をヒートマップで可視化
sns.heatmap(
    cm,                          # 描画対象の混同行列
    annot=True,                  # 各セルに数値を表示
    fmt="d",                     # 数値表示は整数形式
    cmap="Blues",                # 配色テーマ
    xticklabels=target_names,    # 横軸（予測クラス）のラベル
    yticklabels=target_names     # 縦軸（真クラス）のラベル
)
# 横軸ラベルを設定
plt.xlabel("Predicted")
# 縦軸ラベルを設定
plt.ylabel("True")
# グラフタイトルを設定
plt.title("Confusion Matrix")
# レイアウトを自動調整
plt.tight_layout()
# プロットを描画
plt.show()

# ============================================================
# 6. ツリー構造の可視化（第1木 / 上位3段に限定）
# ============================================================
# 図のサイズを指定して新規Figureを作成
plt.figure(figsize=(20, 9))
# ランダムフォレスト内の最初の決定木を可視化
plot_tree(
    rf_model.estimators_[0],     # 描画対象の決定木（インデックス0番目）
    max_depth=3,                 # 描画する最大深度（可読性確保のため上位3段に制限）
    feature_names=X_sel.columns, # 特徴量名（ノード分岐条件に表示）
    class_names=target_names,    # クラス名（葉ノードに表示）
    filled=True,                 # クラスに応じてノードを塗り分け
    rounded=True,                # ノードの角を丸める
    fontsize=10                  # 文字サイズ
)
# グラフタイトルを設定
plt.title("Visualization of a Single Decision Tree (Top 3 Levels)")
# プロットを描画
plt.show()

# ============================================================
# 7. SHAPによる特徴量重要度分析
# ============================================================
# ツリーモデル専用の高速SHAP説明器を初期化
explainer = shap.TreeExplainer(rf_model)
# テストデータに対するSHAP値を計算
shap_values = explainer.shap_values(X_test)

# SHAP値の返り値形状はライブラリバージョンに依存するため両形式に対応
if isinstance(shap_values, list):
    # 旧形式: クラス毎のリスト構造の場合（クラス1=benignを抽出）
    sv_class1 = shap_values[1]
else:
    # 新形式: (n_samples, n_features, n_classes) の3次元配列の場合
    sv_class1 = shap_values[:, :, 1]

# --- 7.1 特徴量重要度（Bar Plot） ---
# SHAP値の平均絶対値を棒グラフで可視化
shap.summary_plot(
    sv_class1,        # クラス1に対するSHAP値
    X_test,           # 対応する特徴量データ
    plot_type="bar",  # プロット種別を棒グラフに指定
    show=True         # 即座に描画
)

# --- 7.2 SHAP値の分布（Beeswarm Plot） ---
# 各サンプルのSHAP値分布をビースウォームプロットで可視化
shap.summary_plot(
    sv_class1,        # クラス1に対するSHAP値
    X_test,           # 対応する特徴量データ
    show=True         # 即座に描画
)

# パイプライン完了メッセージを表示
print("\n--- パイプライン実行完了 ---")
